# Решения: CI и корреляция

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('startup_ab.csv')
df = pd.read_csv(CSV_PATH)
df['variant_b'] = (df['variant'] == 'B').astype(int)


In [ ]:
rng = np.random.default_rng(380)
a = df[df['variant'] == 'A']['converted'].to_numpy()
b = df[df['variant'] == 'B']['converted'].to_numpy()
boot_diffs = []
for _ in range(2500):
    a_s = rng.choice(a, size=len(a), replace=True)
    b_s = rng.choice(b, size=len(b), replace=True)
    boot_diffs.append(float(b_s.mean() - a_s.mean()))
ci_low, ci_high = np.quantile(boot_diffs, [0.025, 0.975])
CI_NOTE = (
    '95% CI — диапазон правдоподобных значений эффекта при выбранной процедуре. '
    'Если ноль вне интервала, эффект статистически совместим с отличием от нуля.'
)
corr_pages = float(df['pages_viewed'].corr(df['converted']))
corr_time = float(df['session_minutes'].corr(df['converted']))
CAUSE_NOTE = (
    'Высокая корреляция не доказывает причинность: на обе переменные может влиять скрытый фактор '
    '(например, качество трафика или намерение пользователя купить).' 
)
b_conv = b
boot_b = [float(rng.choice(b_conv, size=len(b_conv), replace=True).mean()) for _ in range(2500)]
ci_b = tuple(np.quantile(boot_b, [0.025, 0.975]))
rev = df['order_value'].to_numpy()
boot_rev = [float(rng.choice(rev, size=len(rev), replace=True).mean()) for _ in range(2500)]
ci_rev = tuple(np.quantile(boot_rev, [0.025, 0.975]))
rows = []
for dev in ['desktop', 'mobile']:
    part = df[df['device'] == dev]
    rows.append({'device': dev, 'corr_pages_conv': float(part['pages_viewed'].corr(part['converted']))})
corr_table = pd.DataFrame(rows)
LIMIT_NOTE = (
    'Даже при CI и корреляциях остаются ограничения: синтетические данные, возможные скрытые факторы, '
    'и линейная связь может быть только приближением.'
)
print((round(ci_low, 4), round(ci_high, 4)), round(corr_pages, 4), round(corr_time, 4))